# Lyric Engine â€” Kaggle Training Notebook (T4 x2)

**Repo:** https://github.com/SMXFREEZE/lyric-engine  
**Runtime:** GPU T4 x2 (32 GB VRAM total) â€” Settings â†’ Accelerator â†’ GPU T4 x2

### Stages
1. Check GPU (multi-GPU)
2. Install dependencies
3. Clone / update repo from GitHub
4. Config (tokens, model, genre) â€” **fp16, no quantization needed on 32 GB**
5. Scrape lyrics from Genius API
6. Build style vectors
7. **Viral DNA â€” scrape 55-country charts â†’ compute trend signal**
8. Verify pipeline
9. Load model + LoRA (fp16, device_map=auto)
10. Train (with viral conditioning vector)
11. Test generation
12. **Generate full song (MusicGen large + Bark vocals)**
13. Save checkpoint

In [ ]:
# -- 1. Check GPU --
import torch

if not torch.cuda.is_available():
    print('NO GPU â€” go to Settings â†’ Accelerator â†’ GPU T4 x2')
    raise SystemExit('Enable GPU first')

n_gpus = torch.cuda.device_count()
total_vram = 0
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    total_vram += vram
    print(f'GPU {i}: {name}  ({vram:.1f} GB)')

print(f'\nTotal VRAM : {total_vram:.1f} GB')
print(f'GPUs       : {n_gpus}')

if total_vram >= 28:
    print('âœ“ T4 x2 â€” will use fp16, no quantization needed')
elif total_vram >= 14:
    print('âš  Single T4 â€” will use 4-bit quantization')
else:
    print('âš  Low VRAM â€” switching to 4-bit + small model')

!nvidia-smi

In [ ]:
# -- 2. Install dependencies --

# Core ML + training (must succeed)
!pip install -q transformers peft accelerate bitsandbytes trl datasets

# NLP utilities
!pip install -q pronouncing sentence-transformers textblob

# Utilities
!pip install -q fastapi uvicorn rich typer python-dotenv pydub soundfile

# Chart scraping (optional)
!pip install -q billboard.py 2>/dev/null || echo 'billboard.py skipped'

# Audio generation (optional - only needed for cell 12)
!pip install -q audiocraft suno-bark 2>/dev/null || echo 'Audio libs skipped (not needed for training)'

print('All dependencies installed.')


In [ ]:
# -- 3. Clone / update repo --
import os, sys

REPO = 'https://github.com/SMXFREEZE/lyric-engine'
DEST = '/kaggle/working/lyric-engine'

if os.path.exists(f'{DEST}/.git'):
    print('Already cloned - pulling latest main...')
    !git -C {DEST} fetch origin
    !git -C {DEST} checkout main
    !git -C {DEST} pull origin main
else:
    print('Cloning repo...')
    !git clone -b main {REPO} {DEST}

%cd {DEST}
print('Branch:',)
!git rev-parse --abbrev-ref HEAD
print('Latest commits:')
!git log --oneline -5

sys.path.insert(0, '.')

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/style_vectors', exist_ok=True)
print('Checkpoints ->', CHECKPOINT_DIR)


In [ ]:
# -- 4. Config --
# Add GENIUS_TOKEN and HF_TOKEN as Kaggle Secrets (eye icon, left sidebar)

import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GENIUS_TOKEN = secrets.get_secret('GENIUS_TOKEN')
HF_TOKEN     = secrets.get_secret('HF_TOKEN')

os.environ['GENIUS_TOKEN']            = GENIUS_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

import torch
total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory / 1e9
    for i in range(torch.cuda.device_count())
)

USE_4BIT      = total_vram < 20          # False on T4 x2 (32 GB)
MUSICGEN_SIZE = 'large' if total_vram >= 20 else 'small'
LORA_RANK     = 64  if total_vram >= 20 else 32
BATCH_SIZE    = 4   if total_vram >= 20 else 2
GRAD_ACCUM    = 4   if total_vram >= 20 else 8   # eff batch = 16

# v0.1 matches checkpoint vocab -- NOT Instruct, avoids format conflicts
BASE_MODEL = 'mistralai/Mistral-7B-v0.1'

GENRE         = 'trap'
NUM_EPOCHS    = 10        # up from 3 -> ~10k steps on 5k songs
LR            = 5e-5     # down from 2e-4 -> stable convergence, less forgetting
MAX_LENGTH    = 512      # packed context window per chunk
SAVE_STEPS    = 500      # step-based checkpoint
SAMPLE_STEPS  = 500      # print generation sample every N steps
MAX_PER_ARTIST = 5

CHECKPOINT_DIR    = f'/kaggle/working/checkpoints/{GENRE}'
DATA_PATH         = f'data/raw/{GENRE}.jsonl'
VIRAL_VECTOR_PATH = 'data/viral_vector.npy'
TRAINING_MODE     = 'ccl'

MAX_SONGS_PER_ARTIST = 50
SCRAPE_CHARTS        = True
HF_DATASET_REPO      = 'SMXFREEZE/lyric-engine-data'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Base model    : {BASE_MODEL}')
print(f'Genre         : {GENRE}')
print(f'Total VRAM    : {total_vram:.1f} GB')
print(f'4-bit quant   : {USE_4BIT}')
print(f'LoRA rank     : {LORA_RANK}')
print(f'Eff. batch    : {BATCH_SIZE * GRAD_ACCUM}')
print(f'Epochs        : {NUM_EPOCHS}')
print(f'LR            : {LR}')
print(f'Save every    : {SAVE_STEPS} steps')
print(f'Sample every  : {SAMPLE_STEPS} steps')
print('Genius token  :', 'ok' if GENIUS_TOKEN else 'MISSING')
print('HF token      :', 'ok' if HF_TOKEN else 'MISSING')

In [ ]:
# -- 5. Load lyrics from Genius Song Lyrics + Spotify datasets --
# Attach both datasets in Kaggle sidebar -> Add Data:
#   1. "Genius Song Lyrics" by carlosgdcj
#   2. "900K Spotify" by devdope

import json, os, re
import pandas as pd
from pathlib import Path

out_path = f'data/raw/{GENRE}.jsonl'
os.makedirs('data/raw', exist_ok=True)

TRAP_ARTISTS = {
    'future', 'young thug', 'gunna', 'lil baby', '21 savage', 'roddy ricch',
    'offset', 'playboi carti', 'travis scott', 'metro boomin', 'lil uzi vert',
    'dababy', 'polo g', 'lil durk', 'nba youngboy', 'juice wrld', 'pop smoke',
    'trippie redd', 'ynw melly', 'lil tecca', 'kodak black', 'a boogie wit da hoodie',
    'nav', 'quavo', 'takeoff', 'migos', 'moneybagg yo', 'key glock',
    'pooh shiesty', 'big30', 'rylo rodriguez', 'nle choppa',
}

TRAP_TAGS = {'trap', 'rap', 'hip hop', 'hip-hop', 'drill', 'mumble rap'}

def is_trap_row(artist, genre_tag):
    if genre_tag and any(g in str(genre_tag).lower() for g in TRAP_TAGS):
        return True
    return any(a in str(artist).lower() for a in TRAP_ARTISTS)

def clean_lyrics(raw):
    text = re.sub(r'\[.*?\]', '', str(raw))
    text = text.strip()
    return text

songs = []

csv_files = sorted(Path('/kaggle/input').rglob('*.csv'))
print(f'CSV files found: {len(csv_files)}')
for f in csv_files:
    print(f'  {f}')

if not csv_files:
    raise RuntimeError(
        'No dataset attached!\n'
        'Kaggle sidebar -> Add Data -> search "Genius Song Lyrics" by carlosgdcj'
    )

for fpath in csv_files:
    try:
        print(f'\nReading {fpath.name}...')
        df = pd.read_csv(fpath, on_bad_lines='skip')
        print(f'  Shape: {df.shape}')
        print(f'  Columns: {list(df.columns)}')

        cols_lower = {c.lower(): c for c in df.columns}

        lyric_col  = cols_lower.get('lyrics') or cols_lower.get('lyric') or cols_lower.get('text')
        artist_col = cols_lower.get('artist') or cols_lower.get('artist_name')
        title_col  = cols_lower.get('title') or cols_lower.get('song') or cols_lower.get('song_name')
        genre_col  = cols_lower.get('tag') or cols_lower.get('genre')
        lang_col   = cols_lower.get('language')

        if lyric_col is None:
            print(f'  No lyrics column - skipping')
            continue

        print(f'  lyrics={lyric_col}, artist={artist_col}, title={title_col}, genre={genre_col}')

        if lang_col:
            df = df[df[lang_col].str.lower().fillna('') == 'en']
            print(f'  After English filter: {len(df)} rows')

        for _, row in df.iterrows():
            lyrics = clean_lyrics(row[lyric_col])
            if len(lyrics.split()) < 80:
                continue

            artist    = str(row[artist_col]) if artist_col else 'unknown'
            title     = str(row[title_col])  if title_col  else 'unknown'
            genre_tag = str(row[genre_col])  if genre_col  else ''

            if GENRE == 'trap' and not is_trap_row(artist, genre_tag):
                continue

            songs.append({'artist': artist, 'title': title, 'genre': GENRE, 'lyrics': lyrics})

            if len(songs) >= 15000:
                break

        print(f'  Matched so far: {len(songs)} songs')
        if len(songs) >= 15000:
            break

    except Exception as e:
        print(f'  Error reading {fpath.name}: {e}')

with open(out_path, 'w', encoding='utf-8') as f:
    for r in songs:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f'\nTotal songs saved: {len(songs)}')
print(f'Saved to: {out_path}')
if len(songs) < 1000:
    print('WARNING: few songs - check dataset is attached and has a lyrics column')


In [ ]:
# -- 6. Build style vectors --
from src.data.style_extractor import build_style_vectors

print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=2,
)
print(f'Style vectors built for {len(style_vectors)} artists')

In [ ]:
# -- 7. Viral DNA â€” scrape 55-country charts + compute trend conditioning vector --
import json
import numpy as np

VIRAL_VECTOR_PATH = 'data/viral_conditioning.npy'
VIRAL_REPORT_PATH = 'data/viral_report.json'

if SCRAPE_CHARTS:
    print('Scraping charts from Billboard / Deezer / Spotify CSV / iTunes...')
    print('(Covers 55+ countries including Morocco, all of Africa, Asia, LatAm, Europe)\n')

    try:
        from src.data.chart_scraper import run_weekly, compute_viral_scores
        from src.data.viral_analyzer import run_analysis, build_conditioning_vector

        # 1. Fetch charts (this hits free public APIs â€” no login needed)
        chart_data = run_weekly(output_dir='data/charts')
        print(f'  Charts fetched: {len(chart_data)} country/source entries')

        # 2. Compute viral scores (songs charting in many countries = viral DNA)
        viral_scores = compute_viral_scores(chart_data)
        top_viral = sorted(viral_scores.items(), key=lambda x: x[1], reverse=True)[:20]
        print('\nTop 20 globally viral tracks:')
        for title, score in top_viral:
            print(f'  [{score:5.1f}] {title}')

        # 3. Run full viral analysis (region profiles, trend velocity, universal DNA)
        report = run_analysis(chart_data, output_dir='data/viral_analysis')

        # 4. Build 32-dim conditioning vector for model training
        viral_vec = build_conditioning_vector(report)
        np.save(VIRAL_VECTOR_PATH, viral_vec)

        with open(VIRAL_REPORT_PATH, 'w') as f:
            json.dump(report, f, indent=2, default=str)

        print(f'\nViral conditioning vector shape : {viral_vec.shape}')
        print(f'Saved to: {VIRAL_VECTOR_PATH}')
        print('\nRegion DNA profiles:')
        for region, profile in report.get('region_profiles', {}).items():
            print(f'  {region:15s}: BPM={profile.get("avg_bpm", "?"):.0f}  energy={profile.get("avg_energy", "?"):.2f}')

    except Exception as e:
        print(f'Chart scraping error: {e}')
        print('Falling back to neutral conditioning vector...')
        viral_vec = np.zeros(32, dtype=np.float32)
        np.save(VIRAL_VECTOR_PATH, viral_vec)

else:
    print('SCRAPE_CHARTS=False â€” loading saved vector or using neutral...')
    if os.path.exists(VIRAL_VECTOR_PATH):
        viral_vec = np.load(VIRAL_VECTOR_PATH)
        print(f'Loaded: {VIRAL_VECTOR_PATH}')
    else:
        viral_vec = np.zeros(32, dtype=np.float32)
        np.save(VIRAL_VECTOR_PATH, viral_vec)
        print('Using neutral zero vector')

print(f'\nViral conditioning vector (first 8 dims): {viral_vec[:8].round(3)}')
print('Viral DNA ready âœ“')

In [ ]:
# -- 7. Verify pipeline --
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

with open(f'data/raw/{GENRE}.jsonl') as f:
    sample = json.loads(f.readline())

lines = sample['lyrics'].splitlines()[:8]
print(f"Artist : {sample['artist']}")
print(f"Title  : {sample['title']}")
print()

scheme = detect_scheme(lines)
print(f"Rhyme scheme  : {scheme['scheme_str']} ({scheme['scheme_type']})")
print(f"Rhyme density : {scheme['rhyme_density']}")
print()

for em in score_lyrics('\n'.join(lines)):
    ann = annotate_line(em.text)
    print(f"  [{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables:2d}syl] {em.text[:60]}")

print('\nPipeline OK âœ“')

In [ ]:
# -- 9. Load base model + LoRA --
# modules_to_save=['embed_tokens','lm_head'] saves the resized embeddings
# in every checkpoint, fixing vocab mismatch at inference.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

from configs.genres import SPECIAL_TOKENS
num_added = tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})
print(f'Special tokens added: {num_added}  (vocab size now: {len(tokenizer)})')

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
) if USE_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    torch_dtype=None if USE_4BIT else torch.float16,
    device_map='auto',
    low_cpu_mem_usage=True,
)

model.resize_token_embeddings(len(tokenizer))
print(f'Embeddings resized to: {len(tokenizer)}')

if USE_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    # Saves resized embed_tokens + lm_head -- fixes inference vocab mismatch
    modules_to_save=['embed_tokens', 'lm_head'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

for i in range(torch.cuda.device_count()):
    used_gb  = torch.cuda.memory_allocated(i) / 1e9
    total_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used_gb:.1f}/{total_gb:.1f} GB')

print('\nModel ready.')

In [ ]:
# -- 10. Train: packed sequences + step checkpoints + tokenizer saving --
# PackedLyricsDataset concatenates songs end-to-end into MAX_LENGTH chunks.
# Zero padding -> ~3x more signal per GPU step.
# Tokenizer saved alongside model at every checkpoint.

import os, json, math, time
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup


class PackedLyricsDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length, max_per_artist=None):
        self.max_length = max_length
        self.chunks = []

        records = []
        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        print(f'Loaded {len(records)} records')

        if max_per_artist:
            from collections import Counter
            counts = Counter()
            filtered = []
            for r in records:
                artist = r.get('artist', 'unknown')
                if counts[artist] < max_per_artist:
                    filtered.append(r)
                    counts[artist] += 1
            print(f'After per-artist cap: {len(filtered)} records')
            records = filtered

        eos = tokenizer.eos_token_id
        buf = []
        for r in records:
            text = r.get('text') or r.get('lyrics') or ''
            if not text.strip():
                continue
            buf.extend(tokenizer.encode(text, add_special_tokens=False))
            buf.append(eos)

        print(f'Token buffer: {len(buf):,} tokens')
        for start in range(0, len(buf) - max_length, max_length):
            self.chunks.append(buf[start : start + max_length])
        print(f'Packed chunks: {len(self.chunks):,} x {max_length} tokens')

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return {'input_ids': ids, 'labels': ids.clone()}


def save_checkpoint(model, tokenizer, step, ckpt_dir, best_ckpt=None):
    path = os.path.join(ckpt_dir, f'step_{step}')
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)   # always save tokenizer
    print(f'  Checkpoint -> {path}')
    # Disk cleanup: remove all old checkpoints EXCEPT best and current
    import shutil, glob
    all_ckpts = sorted(glob.glob(os.path.join(ckpt_dir, 'step_*')))
    keep = {path, best_ckpt} - {None}
    for c in all_ckpts:
        if c not in keep and os.path.isdir(c):
            shutil.rmtree(c)
            print(f'  Deleted old checkpoint: {c}')
    return path


def sample_generation(model, tokenizer, step, device):
    model.eval()
    prompt = '[GENRE:trap] [ARC:SETUP] [SECTION:VERSE]\n'
    ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=40, do_sample=True,
            temperature=0.85, top_p=0.92, repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    print(f'\n-- Sample @ step {step} --')
    print(text.strip() or '(empty -- model still warming up)')
    print('-' * 40)
    model.train()


dataset    = PackedLyricsDataset(DATA_PATH, tokenizer, MAX_LENGTH, MAX_PER_ARTIST)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)

optimizer    = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=0.01,
)
total_steps  = math.ceil(len(dataset) / BATCH_SIZE) * NUM_EPOCHS
warmup_steps = min(200, total_steps // 10)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f'Chunks/epoch : {len(dataset):,}')
print(f'Total steps  : {total_steps:,}')
print(f'Warmup steps : {warmup_steps}')
print()

model.train()
model.gradient_checkpointing_enable()

device      = next(p for p in model.parameters() if p.requires_grad).device
global_step = 0
best_loss   = float('inf')
best_ckpt   = None
accum_loss  = 0.0
optimizer.zero_grad()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    t0 = time.time()

    for batch_idx, batch in enumerate(dataloader, 1):
        input_ids = batch['input_ids'].to(device)
        labels    = batch['labels'].to(device)

        out  = model(input_ids=input_ids, labels=labels)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()

        if batch_idx % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            global_step += 1
            step_loss    = accum_loss * GRAD_ACCUM
            accum_loss   = 0.0
            epoch_loss  += step_loss

            if global_step % 50 == 0:
                elapsed = time.time() - t0
                eta_min = elapsed / global_step * (total_steps - global_step) / 60
                print(f'  ep {epoch}/{NUM_EPOCHS}  step {global_step}/{total_steps}'
                      f'  loss {step_loss:.4f}'
                      f'  lr {scheduler.get_last_lr()[0]:.2e}'
                      f'  ETA {eta_min:.0f}m')

            if global_step % SAMPLE_STEPS == 0:
                sample_generation(model, tokenizer, global_step, str(device))

            if global_step % SAVE_STEPS == 0:
                ckpt = save_checkpoint(model, tokenizer, global_step, CHECKPOINT_DIR, best_ckpt)
                if step_loss < best_loss:
                    best_loss = step_loss
                    best_ckpt = ckpt

    avg = epoch_loss / max(len(dataloader) // GRAD_ACCUM, 1)
    print(f'\n[Epoch {epoch}] avg_loss={avg:.4f}  time={(time.time()-t0)/60:.1f}m\n')

final_ckpt = save_checkpoint(model, tokenizer, global_step, CHECKPOINT_DIR)
if not best_ckpt:
    best_ckpt = final_ckpt

print('\nDone.')
print(f'Best  : {best_ckpt}  (loss {best_loss:.4f})')
print(f'Final : {final_ckpt}')

In [ ]:
# -- 11. Test generation â€” FULL SONG (~70 lines) --
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from peft import PeftModel
from src.inference.engine import LyricsEngine, SongMemory

ckpt_path = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'

print(f'Loading checkpoint from {ckpt_path}...')
tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

# Determine device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

base_mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, load_in_4bit=True, device_map='auto'
)
trained_mdl = PeftModel.from_pretrained(base_mdl, ckpt_path)

engine = LyricsEngine(trained_mdl, tok, device=device, beam_size=8)

# Create song memory with CCL format enabled
memory = SongMemory(
    genre=GENRE,
    rhyme_scheme='AABB',
    target_syllables=10,
    mood='dark' if GENRE in ('trap', 'drill', 'alt_emo') else 'hype',
    use_ccl_format=True,  # Use brain-inspired CCL format
)

print(f'\n{"="*60}')
print(f'GENERATING FULL {GENRE.upper()} SONG (~70 lines)')
print(f'Training: CCL (Cortical Creative Loop) format')
print(f'{"="*60}\n')

# Generate full song structure:
# Intro (4) + Verse1 (12) + PreChorus (4) + Chorus (8) + Verse2 (12) +
# PreChorus (4) + Chorus (8) + Bridge (6) + FinalChorus (8) + Outro (4) = 70 lines
song_structure = [
    ("INTRO",      "[SETUP]",   4),
    ("VERSE",      "[SETUP]",   12),
    ("PRECHORUS",  "[BUILD]",   4),
    ("CHORUS",     "[RELEASE]", 8),
    ("VERSE",      "[BUILD]",   12),
    ("PRECHORUS",  "[BUILD]",   4),
    ("CHORUS",     "[RELEASE]", 8),
    ("BRIDGE",     "[REFRAME]", 6),
    ("CHORUS",     "[PEAK]",    8),
    ("OUTRO",      "[OUTRO]",   4),
]

# Generate the full song
full_song = engine.generate_full_song(memory, structure=song_structure)

# Print the complete song
print(f'\n{"="*60}')
print(f'COMPLETE {GENRE.upper()} SONG')
print(f'{"="*60}\n')

total_lines = 0
for section_key, lines in full_song.items():
    section_name = "".join(c for c in section_key if not c.isdigit()).upper()
    print(f'\n[{section_name}]')
    for i, line in enumerate(lines, 1):
        print(f'  {line}')
        total_lines += 1

print(f'\n{"="*60}')
print(f'Total lines generated: {total_lines}')
print(f'{"="*60}')

# Store for audio generation cell
verse = full_song.get('verse1', []) + full_song.get('verse2', [])

In [ ]:
# -- 12. Generate full song (MusicGen large + Bark vocals + final MP3) --
# Uses the full song generated in cell 11 + the trained genre style

from src.audio.song_assembler import SongAssembler, SongSpec

# Use the full_song dict from cell 11 (generated by generate_full_song)
# Map section keys to audio sections
generated_lyrics = {
    'verse1':    full_song.get('verse1', []),
    'chorus':    full_song.get('chorus1', []),
    'verse2':    full_song.get('verse2', []),
    'chorus2':   full_song.get('chorus2', []),
    'bridge':    full_song.get('bridge1', []),
    'chorus3':   full_song.get('chorus3', []),
    'intro':     full_song.get('intro1', []),
    'outro':     full_song.get('outro1', []),
    'prechorus': full_song.get('prechorus1', []),
}

# Filter out empty sections
generated_lyrics = {k: v for k, v in generated_lyrics.items() if v}

spec = SongSpec(
    title=f'AI_{GENRE.upper()}_FULL_001',
    genre=GENRE,
    bpm=140 if GENRE in ('trap', 'drill') else 120,
    mood='dark' if GENRE in ('trap', 'drill', 'alt_emo') else 'hype',
    language='en',
    voice_idx=0,
    lyrics=generated_lyrics,
    sections=[
        ('intro',     8),
        ('verse1',    16),
        ('prechorus', 8),
        ('chorus',    12),
        ('verse2',    16),
        ('prechorus', 8),
        ('chorus2',   12),
        ('bridge',    10),
        ('chorus3',   12),
        ('outro',     8),
    ],
    output_dir='/kaggle/working/songs',
)

print(f'Assembling FULL song: {spec.title}')
print(f'MusicGen size : {MUSICGEN_SIZE}  (large = best quality, uses ~6 GB VRAM)')
print(f'Bark          : small model (vocal synthesis, ~2 GB VRAM)')
print(f'Total sections: {[s[0] for s in spec.sections]}')
print(f'Lyrics sections: {list(generated_lyrics.keys())}')
print(f'Total lyric lines: {sum(len(v) for v in generated_lyrics.values())}')
print()

# Free LLM VRAM before loading audio models
import gc
try:
    del base_mdl, trained_mdl, engine
    gc.collect()
    torch.cuda.empty_cache()
    print('LLM unloaded to free VRAM for audio models')
except Exception:
    pass

# Show VRAM state before audio generation
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used:.1f}/{total:.1f} GB free after LLM unload')

assembler = SongAssembler(
    musicgen_size=MUSICGEN_SIZE,
    bark_small=True,
    device='auto',
)

song = assembler.generate_song(spec, skip_vocals=False, skip_instrumental=False)

print(f'\n{"="*60}')
print(f'FULL SONG generated successfully!')
print(f'Output: {song.output_path}')
print(f'Duration: {len(song.mixed) / 44100:.1f}s')
print(f'\nFiles in /kaggle/working/songs/:')
import os, pathlib
for p in sorted(pathlib.Path('/kaggle/working/songs').rglob('*')):
    if p.is_file():
        size = p.stat().st_size / 1e6
        print(f'  {p.name:50s} {size:.1f} MB')
print('\nDownload from: Kaggle â†’ Output tab (right sidebar) â†’ songs/')

In [ ]:
# -- 13. Package best checkpoint for download + Kaggle dataset re-upload --
import os, zipfile

print(f'Best checkpoint: {best_ckpt}')
print()
for fname in sorted(os.listdir(best_ckpt)):
    sz = os.path.getsize(os.path.join(best_ckpt, fname))
    print(f'  {fname:50s}  {sz/1e6:6.1f} MB')

zip_path = f'/kaggle/working/lyric_engine_{GENRE}_best.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(best_ckpt):
        zf.write(os.path.join(best_ckpt, fname), fname)

print(f'\nZip: {zip_path}  ({os.path.getsize(zip_path)/1e9:.2f} GB)')
print()
print('Next steps:')
print('  1. Kaggle -> Output tab -> Download the zip')
print('  2. Unzip locally')
print('  3. Kaggle Datasets -> Create dataset -> Upload unzipped folder')
print('  4. run_ai_kaggle.ipynb -> Add that dataset as input')
print()
print('Tokenizer is inside the checkpoint -> vocab mismatch is FIXED.')